In [2]:
# Featuring Engineeri

In [7]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

transactions = pd.read_csv(r"D:\Machine Learning\upi_fraud_detection\data\processed\transactions_clean.csv")
users = pd.read_csv(r"D:\Machine Learning\upi_fraud_detection\data\processed\users_clean.csv")

transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])

print("Transactions:", transactions.shape)
print("Users:", users.shape)

Transactions: (15000, 19)
Users: (1000, 8)


In [8]:
# Step 2 — User profile join
df = transactions.merge(
    users,
    left_on="sender_id",
    right_on="user_id",
    how="left"
)

print(df.shape)
df.head()

(15000, 27)


,transaction_id,timestamp,sender_id,receiver_id,amount,merchant_category,transaction_type,city,latitude,longitude,device_id,upi_channel,fraud_label,fraud_scenario,date,hour,day_of_week,is_weekend,is_late_night,user_id,profile_type,avg_transaction_amount,daily_transaction_rate,home_city,home_lat,home_lon,registered_device
0,TX0150149,2026-01-01 00:05:06,U100666,R99857,1964.88,Ecommerce,P2M,Patna,25.61338,85.12217,D100666,QR,0,normal,2026-01-01,0,Thursday,False,True,U100666,regular,1672.84,2.48,Patna,25.5941,85.1376,D100666
1,TX0016422,2026-01-01 00:07:23,U100080,R87314,1965.93,Ecommerce,Bill_Payment,Delhi,28.62374,77.16503,D100080,QR,0,normal,2026-01-01,0,Thursday,False,True,U100080,regular,3206.72,2.40,Delhi,28.6139,77.2090,D100080
2,TX0045899,2026-01-01 00:23:11,U100214,R65704,33723.94,Utilities,P2P,Jaipur,26.97108,75.79332,D100214,Mobile_Number,0,normal,2026-01-01,0,Thursday,False,True,U100214,business,23530.58,5.31,Jaipur,26.9124,75.7873,D100214
3,TX0017833,2026-01-01 00:36:36,U100086,R31352,2957.32,Food,P2P,Bengaluru,12.98862,77.59862,D100086,Mobile_Number,0,normal,2026-01-01,0,Thursday,False,True,U100086,regular,4703.06,1.86,Bengaluru,12.9716,77.5946,D100086
4,TX0083922,2026-01-01 00:38:45,U100379,R99128,3617.79,Shopping,Recharge,Hyderabad,17.37540,78.53740,D100379,Collect_Request,0,normal,2026-01-01,0,Thursday,False,True,U100379,regular,2693.02,3.37,Hyderabad,17.3850,78.4867,D100379


In [9]:
# Step 3 — Amount-based features
df["amount_vs_user_avg"] = (
    df["amount"] / df["avg_transaction_amount"]
)

df["amount_deviation"] = (
    df["amount"] - df["avg_transaction_amount"]
)

df["amount_ratio"] = (
    df["amount"] / df["avg_transaction_amount"]
).replace([np.inf, -np.inf], np.nan).fillna(0)

In [10]:
# Step 4 — Time features
df["hour"] = df["timestamp"].dt.hour

df["day_of_week"] = df["timestamp"].dt.dayofweek

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

df["is_late_night"] = (
    (df["hour"] >= 0) & (df["hour"] < 5)
).astype(int)

In [11]:
# Step 5 — Device feature
df["is_new_device"] = (
    df["device_id"] != df["registered_device"]
).astype(int)

In [12]:
# Step 6 — Location feature
def haversine_distance(lat1, lon1, lat2, lon2):

    R = 6371

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arctan2(
        np.sqrt(a),
        np.sqrt(1 - a)
    )

    return R * c


df["distance_from_home_km"] = haversine_distance(
    df["home_lat"],
    df["home_lon"],
    df["latitude"],
    df["longitude"]
)

In [14]:
# Step 7 — User transaction frequency
# Sort data properly
df = df.sort_values(["sender_id", "timestamp"]).copy()

# Make timestamp the index within each user
df["transactions_last_10m"] = 0
df["transactions_last_1h"] = 0
df["transactions_last_24h"] = 0

for user_id, group in df.groupby("sender_id"):
    
    idx = group.index
    temp = group.set_index("timestamp")

    df.loc[idx, "transactions_last_10m"] = (
        temp["transaction_id"]
        .rolling("10min")
        .count()
        .values
    )

    df.loc[idx, "transactions_last_1h"] = (
        temp["transaction_id"]
        .rolling("1h")
        .count()
        .values
    )

    df.loc[idx, "transactions_last_24h"] = (
        temp["transaction_id"]
        .rolling("24h")
        .count()
        .values
    )

print(df[
    [
        "sender_id",
        "timestamp",
        "transactions_last_10m",
        "transactions_last_1h",
        "transactions_last_24h"
    ]
].head(20))

      sender_id           timestamp  transactions_last_10m  \
428     U100000 2026-01-03 13:26:04                      1   
689     U100000 2026-01-05 01:51:05                      1   
2129    U100000 2026-01-14 06:21:25                      1   
2352    U100000 2026-01-15 14:05:07                      1   
2455    U100000 2026-01-16 07:27:50                      1   
3435    U100000 2026-01-22 02:05:05                      1   
3803    U100000 2026-01-24 05:30:30                      1   
4735    U100000 2026-01-29 18:44:28                      1   
5068    U100000 2026-01-31 18:03:23                      1   
5071    U100000 2026-01-31 18:22:47                      1   
9494    U100000 2026-02-27 01:23:27                      1   
10185   U100000 2026-03-02 20:51:35                      1   
10624   U100000 2026-03-05 11:42:59                      1   
12462   U100000 2026-03-16 16:21:59                      1   
12478   U100000 2026-03-16 18:09:12                      1   
237     

In [15]:
# Step 8 — Time Since Previous Transaction
# Sort transactions by user and time
df = df.sort_values(["sender_id", "timestamp"]).copy()

# Calculate time difference from previous transaction
df["time_since_previous_transaction"] = (
    df.groupby("sender_id")["timestamp"]
      .diff()
      .dt.total_seconds()
      / 60
)

# First transaction of each user has no previous transaction
df["time_since_previous_transaction"] = (
    df["time_since_previous_transaction"]
    .fillna(-1)
)

# Check result
print(
    df[
        [
            "sender_id",
            "timestamp",
            "time_since_previous_transaction"
        ]
    ].head(20)
)

      sender_id           timestamp  time_since_previous_transaction
428     U100000 2026-01-03 13:26:04                        -1.000000
689     U100000 2026-01-05 01:51:05                      2185.016667
2129    U100000 2026-01-14 06:21:25                     13230.333333
2352    U100000 2026-01-15 14:05:07                      1903.700000
2455    U100000 2026-01-16 07:27:50                      1042.716667
3435    U100000 2026-01-22 02:05:05                      8317.250000
3803    U100000 2026-01-24 05:30:30                      3085.416667
4735    U100000 2026-01-29 18:44:28                      7993.966667
5068    U100000 2026-01-31 18:03:23                      2838.916667
5071    U100000 2026-01-31 18:22:47                        19.400000
9494    U100000 2026-02-27 01:23:27                     37860.666667
10185   U100000 2026-03-02 20:51:35                      5488.133333
10624   U100000 2026-03-05 11:42:59                      3771.400000
12462   U100000 2026-03-16 16:21:5

In [16]:
# Step 9 — Recipient Behavior
# Count how many times each sender has transacted with each receiver
df["recipient_transaction_count"] = (
    df.groupby(["sender_id", "receiver_id"])
      .cumcount()
)

# New recipient = first transaction with this receiver
df["is_new_recipient"] = (
    df["recipient_transaction_count"] == 0
).astype(int)

# Check result
print(
    df[
        [
            "sender_id",
            "receiver_id",
            "recipient_transaction_count",
            "is_new_recipient"
        ]
    ].head(20)
)

      sender_id   receiver_id  recipient_transaction_count  is_new_recipient
428     U100000  R_FRAUD_1058                            0                 1
689     U100000        R88146                            0                 1
2129    U100000        R90967                            0                 1
2352    U100000        R25509                            0                 1
2455    U100000        R54357                            0                 1
3435    U100000        R75325                            0                 1
3803    U100000        R21382                            0                 1
4735    U100000        R77357                            0                 1
5068    U100000        R55523                            0                 1
5071    U100000        R89349                            0                 1
9494    U100000        R77720                            0                 1
10185   U100000        R37941                            0                 1

In [18]:
print("New recipients:")
print(df["is_new_recipient"].value_counts())

print("\nRecipient transaction count:")
print(df["recipient_transaction_count"].value_counts().sort_index().head(10))

New recipients:
is_new_recipient
1    14999
0        1
Name: count, dtype: int64

Recipient transaction count:
recipient_transaction_count
0    14999
1        1
Name: count, dtype: int64


In [19]:
# Step 10 — Device Behavior
# Count previous transactions from the same device by each sender
df["device_transaction_count"] = (
    df.groupby(["sender_id", "device_id"])
      .cumcount()
)

# First use of a device by a sender
df["is_new_device"] = (
    df["device_transaction_count"] == 0
).astype(int)

# Check result
print(
    df[
        [
            "sender_id",
            "device_id",
            "device_transaction_count",
            "is_new_device"
        ]
    ].head(20)
)

      sender_id device_id  device_transaction_count  is_new_device
428     U100000   D100000                         0              1
689     U100000   D100000                         1              0
2129    U100000   D100000                         2              0
2352    U100000   D100000                         3              0
2455    U100000   D100000                         4              0
3435    U100000   D100000                         5              0
3803    U100000   D100000                         6              0
4735    U100000   D100000                         7              0
5068    U100000   D100000                         8              0
5071    U100000   D100000                         9              0
9494    U100000   D100000                        10              0
10185   U100000   D100000                        11              0
10624   U100000   D100000                        12              0
12462   U100000   D100000                        13           

In [24]:
print(df.columns.tolist())

['transaction_id', 'timestamp', 'sender_id', 'receiver_id', 'amount', 'merchant_category', 'transaction_type', 'city', 'latitude', 'longitude', 'device_id', 'upi_channel', 'fraud_label', 'fraud_scenario', 'date', 'hour', 'day_of_week', 'is_weekend', 'is_late_night', 'user_id', 'profile_type', 'avg_transaction_amount', 'daily_transaction_rate', 'home_city', 'home_lat', 'home_lon', 'registered_device', 'amount_vs_user_avg', 'amount_deviation', 'amount_ratio', 'is_new_device', 'distance_from_home_km', 'transactions_last_10m', 'transactions_last_1h', 'transactions_last_24h', 'time_since_previous_transaction', 'recipient_transaction_count', 'is_new_recipient', 'device_transaction_count']


In [25]:
#  Step 11 — Location Behavior
# Sort transactions by sender and timestamp
df = df.sort_values(["sender_id", "timestamp"]).copy()

# Count previous transactions from the same city
df["location_transaction_count"] = (
    df.groupby(["sender_id", "city"])
      .cumcount()
)

# Check whether this is a new location for the sender
df["is_new_location"] = (
    df["location_transaction_count"] == 0
).astype(int)

# Previous transaction's latitude and longitude
df["previous_latitude"] = (
    df.groupby("sender_id")["latitude"]
      .shift(1)
)

df["previous_longitude"] = (
    df.groupby("sender_id")["longitude"]
      .shift(1)
)

print(
    df[
        [
            "sender_id",
            "city",
            "latitude",
            "longitude",
            "distance_from_home_km",
            "location_transaction_count",
            "is_new_location",
            "previous_latitude",
            "previous_longitude"
        ]
    ].head(20)
)

      sender_id       city  latitude  longitude  distance_from_home_km  \
428     U100000  Hyderabad  17.39472   78.48280               0.414269   
689     U100000  Hyderabad  17.44129   78.51521               3.026856   
2129    U100000  Hyderabad  17.38443   78.45662               3.191956   
2352    U100000  Hyderabad  17.43150   78.46320               2.495027   
2455    U100000  Hyderabad  17.35805   78.50146               1.567251   
3435    U100000  Hyderabad  17.39705   78.50740               2.196641   
3803    U100000  Hyderabad  17.40454   78.47724               1.004514   
4735    U100000  Hyderabad  17.39127   78.42791               6.238428   
5068    U100000  Hyderabad  17.33361   78.52732               4.312165   
5071    U100000  Hyderabad  17.42333   78.46895               1.884819   
9494    U100000  Hyderabad  17.35528   78.46971               1.803969   
10185   U100000  Hyderabad  17.39091   78.50885               2.350446   
10624   U100000  Hyderabad  17.39712  

In [26]:
# Step 11B — Previous Location Distance
import numpy as np

# Convert latitude and longitude from degrees to radians
lat1 = np.radians(df["previous_latitude"])
lon1 = np.radians(df["previous_longitude"])

lat2 = np.radians(df["latitude"])
lon2 = np.radians(df["longitude"])

# Haversine formula
dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)

c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

# Earth's radius in kilometers
earth_radius_km = 6371

df["distance_from_previous_location_km"] = (
    earth_radius_km * c
)

# First transaction of each user has no previous location
df["distance_from_previous_location_km"] = (
    df["distance_from_previous_location_km"]
    .fillna(0)
)

print(
    df[
        [
            "sender_id",
            "timestamp",
            "city",
            "distance_from_previous_location_km"
        ]
    ].head(20)
)

      sender_id           timestamp       city  \
428     U100000 2026-01-03 13:26:04  Hyderabad   
689     U100000 2026-01-05 01:51:05  Hyderabad   
2129    U100000 2026-01-14 06:21:25  Hyderabad   
2352    U100000 2026-01-15 14:05:07  Hyderabad   
2455    U100000 2026-01-16 07:27:50  Hyderabad   
3435    U100000 2026-01-22 02:05:05  Hyderabad   
3803    U100000 2026-01-24 05:30:30  Hyderabad   
4735    U100000 2026-01-29 18:44:28  Hyderabad   
5068    U100000 2026-01-31 18:03:23  Hyderabad   
5071    U100000 2026-01-31 18:22:47  Hyderabad   
9494    U100000 2026-02-27 01:23:27  Hyderabad   
10185   U100000 2026-03-02 20:51:35  Hyderabad   
10624   U100000 2026-03-05 11:42:59  Hyderabad   
12462   U100000 2026-03-16 16:21:59  Hyderabad   
12478   U100000 2026-03-16 18:09:12  Hyderabad   
237     U100001 2026-01-02 09:05:43    Kolkata   
240     U100001 2026-01-02 09:28:24    Kolkata   
1010    U100001 2026-01-07 03:03:25    Kolkata   
1641    U100001 2026-01-11 06:49:39    Kolkata   


In [27]:
# Step 11C — Travel Speed + Impossible Travel
# Calculate travel speed in km/h
df["travel_speed_kmph"] = np.where(
    df["time_since_previous_transaction"] > 0,
    df["distance_from_previous_location_km"]
    / (df["time_since_previous_transaction"] / 60),
    0
)

# Flag suspiciously high travel speed
df["impossible_travel_flag"] = (
    df["travel_speed_kmph"] > 500
).astype(int)

print(
    df[
        [
            "sender_id",
            "timestamp",
            "distance_from_previous_location_km",
            "time_since_previous_transaction",
            "travel_speed_kmph",
            "impossible_travel_flag"
        ]
    ].head(20)
)

      sender_id           timestamp  distance_from_previous_location_km  \
428     U100000 2026-01-03 13:26:04                            0.000000   
689     U100000 2026-01-05 01:51:05                            6.216036   
2129    U100000 2026-01-14 06:21:25                            8.866657   
2352    U100000 2026-01-15 14:05:07                            5.280303   
2455    U100000 2026-01-16 07:27:50                            9.120630   
3435    U100000 2026-01-22 02:05:05                            4.382175   
3803    U100000 2026-01-24 05:30:30                            3.306764   
4735    U100000 2026-01-29 18:44:28                            5.438308   
5068    U100000 2026-01-31 18:03:23                           12.345633   
5071    U100000 2026-01-31 18:22:47                           11.742936   
9494    U100000 2026-02-27 01:23:27                            7.567245   
10185   U100000 2026-03-02 20:51:35                            5.740128   
10624   U100000 2026-03-0

In [28]:
# Step 12 — User Spending Behavior
# 12.1 Daily Spending
df["daily_spending"] = (
    df.groupby(["sender_id", "date"])["amount"]
      .transform("sum")
)

In [29]:
# 12.2 Daily Transaction Count
df["transaction_count_per_day"] = (
    df.groupby(["sender_id", "date"])["transaction_id"]
      .transform("count")
)

In [30]:
print(
    df[
        [
            "sender_id",
            "date",
            "amount",
            "daily_spending",
            "transaction_count_per_day"
        ]
    ].head(20)
)

      sender_id        date   amount  daily_spending  \
428     U100000  2026-01-03  2582.92         2582.92   
689     U100000  2026-01-05  1327.52         1327.52   
2129    U100000  2026-01-14  1007.41         1007.41   
2352    U100000  2026-01-15   663.43          663.43   
2455    U100000  2026-01-16   800.96          800.96   
3435    U100000  2026-01-22  1036.17         1036.17   
3803    U100000  2026-01-24  1859.02         1859.02   
4735    U100000  2026-01-29   562.08          562.08   
5068    U100000  2026-01-31   849.85         2023.22   
5071    U100000  2026-01-31  1173.37         2023.22   
9494    U100000  2026-02-27  3669.65         3669.65   
10185   U100000  2026-03-02   529.85          529.85   
10624   U100000  2026-03-05  1006.66         1006.66   
12462   U100000  2026-03-16   872.84         1608.97   
12478   U100000  2026-03-16   736.13         1608.97   
237     U100001  2026-01-02   295.12          731.56   
240     U100001  2026-01-02   436.44          73

In [31]:
# Step 13 — Behavioral Deviation Features
# Step 13.1 — Daily spending baseline per user

df = df.sort_values(["sender_id", "timestamp"]).copy()

df["avg_daily_spending"] = (
    df.groupby("sender_id")["daily_spending"]
      .transform("mean")
)

In [32]:
# Step 13.2 — Spending deviation

df["daily_spending_deviation"] = (
    df["daily_spending"] - df["avg_daily_spending"]
)

In [33]:
# Step 13.3 — Spending ratio

df["daily_spending_ratio"] = (
    df["daily_spending"] / df["avg_daily_spending"]
)

In [34]:
print(
    df[
        [
            "sender_id",
            "date",
            "daily_spending",
            "avg_daily_spending",
            "daily_spending_deviation",
            "daily_spending_ratio"
        ]
    ].head(20)
)

      sender_id        date  daily_spending  avg_daily_spending  \
428     U100000  2026-01-03         2582.92         1487.336667   
689     U100000  2026-01-05         1327.52         1487.336667   
2129    U100000  2026-01-14         1007.41         1487.336667   
2352    U100000  2026-01-15          663.43         1487.336667   
2455    U100000  2026-01-16          800.96         1487.336667   
3435    U100000  2026-01-22         1036.17         1487.336667   
3803    U100000  2026-01-24         1859.02         1487.336667   
4735    U100000  2026-01-29          562.08         1487.336667   
5068    U100000  2026-01-31         2023.22         1487.336667   
5071    U100000  2026-01-31         2023.22         1487.336667   
9494    U100000  2026-02-27         3669.65         1487.336667   
10185   U100000  2026-03-02          529.85         1487.336667   
10624   U100000  2026-03-05         1006.66         1487.336667   
12462   U100000  2026-03-16         1608.97         1487.33666

In [35]:
df["high_spending_deviation_flag"] = (
    df["daily_spending_ratio"] >= 2
).astype(int)

In [36]:
print(
    df[
        [
            "sender_id",
            "date",
            "daily_spending",
            "avg_daily_spending",
            "daily_spending_ratio",
            "high_spending_deviation_flag"
        ]
    ].head(30)
)

      sender_id        date  daily_spending  avg_daily_spending  \
428     U100000  2026-01-03         2582.92         1487.336667   
689     U100000  2026-01-05         1327.52         1487.336667   
2129    U100000  2026-01-14         1007.41         1487.336667   
2352    U100000  2026-01-15          663.43         1487.336667   
2455    U100000  2026-01-16          800.96         1487.336667   
3435    U100000  2026-01-22         1036.17         1487.336667   
3803    U100000  2026-01-24         1859.02         1487.336667   
4735    U100000  2026-01-29          562.08         1487.336667   
5068    U100000  2026-01-31         2023.22         1487.336667   
5071    U100000  2026-01-31         2023.22         1487.336667   
9494    U100000  2026-02-27         3669.65         1487.336667   
10185   U100000  2026-03-02          529.85         1487.336667   
10624   U100000  2026-03-05         1006.66         1487.336667   
12462   U100000  2026-03-16         1608.97         1487.33666

In [37]:
# Step 13.4 — User spending volatility

df["user_amount_std"] = (
    df.groupby("sender_id")["amount"]
      .transform("std")
)

df["amount_vs_user_std"] = (
    df["amount_deviation"] / df["user_amount_std"]
)

In [38]:
print(
    df[
        [
            "sender_id",
            "amount",
            "amount_deviation",
            "user_amount_std",
            "amount_vs_user_std"
        ]
    ].head(20)
)

      sender_id   amount  amount_deviation  user_amount_std  \
428     U100000  2582.92           1482.88       856.971103   
689     U100000  1327.52            227.48       856.971103   
2129    U100000  1007.41            -92.63       856.971103   
2352    U100000   663.43           -436.61       856.971103   
2455    U100000   800.96           -299.08       856.971103   
3435    U100000  1036.17            -63.87       856.971103   
3803    U100000  1859.02            758.98       856.971103   
4735    U100000   562.08           -537.96       856.971103   
5068    U100000   849.85           -250.19       856.971103   
5071    U100000  1173.37             73.33       856.971103   
9494    U100000  3669.65           2569.61       856.971103   
10185   U100000   529.85           -570.19       856.971103   
10624   U100000  1006.66            -93.38       856.971103   
12462   U100000   872.84           -227.20       856.971103   
12478   U100000   736.13           -363.91       856.97

In [39]:
# Step 13.5 — Extreme transaction amount flag

df["extreme_amount_flag"] = (
    df["amount_vs_user_std"].abs() >= 2
).astype(int)

In [40]:
print(
    df[
        [
            "sender_id",
            "amount",
            "amount_vs_user_std",
            "extreme_amount_flag"
        ]
    ].head(20)
)

      sender_id   amount  amount_vs_user_std  extreme_amount_flag
428     U100000  2582.92            1.730373                    0
689     U100000  1327.52            0.265447                    0
2129    U100000  1007.41           -0.108090                    0
2352    U100000   663.43           -0.509480                    0
2455    U100000   800.96           -0.348997                    0
3435    U100000  1036.17           -0.074530                    0
3803    U100000  1859.02            0.885654                    0
4735    U100000   562.08           -0.627746                    0
5068    U100000   849.85           -0.291947                    0
5071    U100000  1173.37            0.085569                    0
9494    U100000  3669.65            2.998479                    1
10185   U100000   529.85           -0.665355                    0
10624   U100000  1006.66           -0.108965                    0
12462   U100000   872.84           -0.265120                    0
12478   U1

In [41]:
print("Dataset Shape:", df.shape)

print("\nTotal Features:")
print(len(df.columns))

print("\nFeature Columns:")
print(df.columns.tolist())

Dataset Shape: (15000, 55)

Total Features:
55

Feature Columns:
['transaction_id', 'timestamp', 'sender_id', 'receiver_id', 'amount', 'merchant_category', 'transaction_type', 'city', 'latitude', 'longitude', 'device_id', 'upi_channel', 'fraud_label', 'fraud_scenario', 'date', 'hour', 'day_of_week', 'is_weekend', 'is_late_night', 'user_id', 'profile_type', 'avg_transaction_amount', 'daily_transaction_rate', 'home_city', 'home_lat', 'home_lon', 'registered_device', 'amount_vs_user_avg', 'amount_deviation', 'amount_ratio', 'is_new_device', 'distance_from_home_km', 'transactions_last_10m', 'transactions_last_1h', 'transactions_last_24h', 'time_since_previous_transaction', 'recipient_transaction_count', 'is_new_recipient', 'device_transaction_count', 'location_transaction_count', 'is_new_location', 'previous_latitude', 'previous_longitude', 'distance_from_previous_location_km', 'travel_speed_kmph', 'impossible_travel_flag', 'daily_spending', 'transaction_count_per_day', 'avg_daily_spending

In [50]:
feature_cols = [
    "transactions_last_10m",
    "transactions_last_1h",
    "transactions_last_24h",
    "time_since_previous_transaction",
    "recipient_transaction_count",
    "is_new_recipient",
    "device_transaction_count",
    "is_new_device",
    "location_transaction_count",
    "is_new_location",
    "distance_from_previous_location_km",
    "travel_speed_kmph",
    "impossible_travel_flag",
    "daily_spending",
    "transaction_count_per_day",
    "avg_daily_spending",
    "daily_spending_deviation",
    "daily_spending_ratio",
    "high_spending_deviation_flag",
    "user_amount_std",
    "amount_vs_user_std",
    "extreme_amount_flag"
]

print(df[feature_cols].describe().T)

                                      count          mean           std  \
transactions_last_10m               15000.0  1.001400e+00      0.037392   
transactions_last_1h                15000.0  1.008400e+00      0.091269   
transactions_last_24h               15000.0  1.200933e+00      0.459463   
time_since_previous_transaction     15000.0  7.211642e+03   8668.174678   
recipient_transaction_count         15000.0  6.666667e-05      0.008165   
is_new_recipient                    15000.0  9.999333e-01      0.008165   
device_transaction_count            15000.0  8.833067e+00      7.650684   
is_new_device                       15000.0  8.580000e-02      0.280078   
location_transaction_count          15000.0  8.404467e+00      7.530966   
is_new_location                     15000.0  1.085333e-01      0.311063   
distance_from_previous_location_km  15000.0  8.987556e+01    315.213901   
travel_speed_kmph                   15000.0  7.026684e+00    193.529273   
impossible_travel_flag   

In [51]:
# Step 14 — Check missing values

missing = df.isnull().sum()

print(missing[missing > 0])

previous_latitude     1000
previous_longitude    1000
dtype: int64


In [52]:
df = df.drop(
    columns=["previous_latitude", "previous_longitude"]
)

In [53]:
df["user_amount_std"] = df["user_amount_std"].fillna(0)

df["amount_vs_user_std"] = df["amount_vs_user_std"].fillna(0)

In [54]:
print("Remaining missing values:")

print(
    df.isnull().sum()[df.isnull().sum() > 0]
)

Remaining missing values:
Series([], dtype: int64)


In [55]:
# Step 14 — Historical / Past-only Features
df = df.sort_values(["sender_id", "timestamp"]).copy()

In [56]:
df["historical_avg_amount"] = (
    df.groupby("sender_id")["amount"]
      .transform(lambda x: x.shift(1).expanding().mean())
)

df["historical_avg_amount"] = (
    df["historical_avg_amount"].fillna(df["avg_transaction_amount"])
)

In [57]:
df["historical_amount_std"] = (
    df.groupby("sender_id")["amount"]
      .transform(
          lambda x: x.shift(1).expanding().std()
      )
)

df["historical_amount_std"] = (
    df["historical_amount_std"].fillna(0)
)

In [58]:
df["historical_amount_zscore"] = np.where(
    df["historical_amount_std"] > 0,
    (
        df["amount"] - df["historical_avg_amount"]
    ) / df["historical_amount_std"],
    0
)

In [59]:
print(
    df[
        [
            "sender_id",
            "timestamp",
            "amount",
            "historical_avg_amount",
            "historical_amount_std",
            "historical_amount_zscore"
        ]
    ].head(20)
)

      sender_id           timestamp   amount  historical_avg_amount  \
428     U100000 2026-01-03 13:26:04  2582.92            1100.040000   
689     U100000 2026-01-05 01:51:05  1327.52            2582.920000   
2129    U100000 2026-01-14 06:21:25  1007.41            1955.220000   
2352    U100000 2026-01-15 14:05:07   663.43            1639.283333   
2455    U100000 2026-01-16 07:27:50   800.96            1395.320000   
3435    U100000 2026-01-22 02:05:05  1036.17            1276.448000   
3803    U100000 2026-01-24 05:30:30  1859.02            1236.401667   
4735    U100000 2026-01-29 18:44:28   562.08            1325.347143   
5068    U100000 2026-01-31 18:03:23   849.85            1229.938750   
5071    U100000 2026-01-31 18:22:47  1173.37            1187.706667   
9494    U100000 2026-02-27 01:23:27  3669.65            1186.273000   
10185   U100000 2026-03-02 20:51:35   529.85            1412.034545   
10624   U100000 2026-03-05 11:42:59  1006.66            1338.519167   
12462 

In [60]:
# Step 15 — Historical Daily Spending Features

df = df.sort_values(["sender_id", "timestamp"]).copy()

# Daily spending at user-day level
daily = (
    df.groupby(["sender_id", "date"], as_index=False)
      .agg(
          daily_spending=("amount", "sum"),
          daily_transaction_count=("transaction_id", "count")
      )
)

# Previous day's spending for each user
daily["previous_daily_spending"] = (
    daily.groupby("sender_id")["daily_spending"]
         .shift(1)
)

# Historical average of previous days only
daily["historical_avg_daily_spending"] = (
    daily.groupby("sender_id")["daily_spending"]
         .transform(lambda x: x.shift(1).expanding().mean())
)

# First day fallback
daily["historical_avg_daily_spending"] = (
    daily["historical_avg_daily_spending"]
    .fillna(daily["daily_spending"].median())
)

# Deviation
daily["daily_spending_deviation"] = (
    daily["daily_spending"]
    - daily["historical_avg_daily_spending"]
)

# Ratio
daily["daily_spending_ratio"] = (
    daily["daily_spending"]
    / daily["historical_avg_daily_spending"].replace(0, np.nan)
)

daily["daily_spending_ratio"] = (
    daily["daily_spending_ratio"].fillna(1)
)

# High spending flag
daily["high_spending_deviation_flag"] = (
    daily["daily_spending_ratio"] >= 2
).astype(int)

# Merge back to transaction-level dataframe
df = df.drop(
    columns=[
        "daily_spending",
        "transaction_count_per_day",
        "avg_daily_spending",
        "daily_spending_deviation",
        "daily_spending_ratio",
        "high_spending_deviation_flag"
    ],
    errors="ignore"
)

df = df.merge(
    daily[
        [
            "sender_id",
            "date",
            "daily_spending",
            "daily_transaction_count",
            "historical_avg_daily_spending",
            "previous_daily_spending",
            "daily_spending_deviation",
            "daily_spending_ratio",
            "high_spending_deviation_flag"
        ]
    ],
    on=["sender_id", "date"],
    how="left"
)

In [61]:
print(
    df[
        [
            "sender_id",
            "date",
            "amount",
            "daily_spending",
            "historical_avg_daily_spending",
            "daily_spending_ratio",
            "high_spending_deviation_flag"
        ]
    ].head(20)
)

   sender_id        date   amount  daily_spending  \
0    U100000  2026-01-03  2582.92         2582.92   
1    U100000  2026-01-05  1327.52         1327.52   
2    U100000  2026-01-14  1007.41         1007.41   
3    U100000  2026-01-15   663.43          663.43   
4    U100000  2026-01-16   800.96          800.96   
5    U100000  2026-01-22  1036.17         1036.17   
6    U100000  2026-01-24  1859.02         1859.02   
7    U100000  2026-01-29   562.08          562.08   
8    U100000  2026-01-31   849.85         2023.22   
9    U100000  2026-01-31  1173.37         2023.22   
10   U100000  2026-02-27  3669.65         3669.65   
11   U100000  2026-03-02   529.85          529.85   
12   U100000  2026-03-05  1006.66         1006.66   
13   U100000  2026-03-16   872.84         1608.97   
14   U100000  2026-03-16   736.13         1608.97   
15   U100001  2026-01-02   295.12          731.56   
16   U100001  2026-01-02   436.44          731.56   
17   U100001  2026-01-07   169.72          169

In [62]:
# Step 16 — Final Feature Cleanup

drop_columns = [
    "previous_latitude",
    "previous_longitude",
    "avg_daily_spending",
    "user_amount_std"
]

df = df.drop(
    columns=drop_columns,
    errors="ignore"
)

# Final missing-value check
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nFinal Shape:", df.shape)

print("\nFinal Columns:")
print(df.columns.tolist())

Missing values:
previous_daily_spending    1081
dtype: int64

Final Shape: (15000, 56)

Final Columns:
['transaction_id', 'timestamp', 'sender_id', 'receiver_id', 'amount', 'merchant_category', 'transaction_type', 'city', 'latitude', 'longitude', 'device_id', 'upi_channel', 'fraud_label', 'fraud_scenario', 'date', 'hour', 'day_of_week', 'is_weekend', 'is_late_night', 'user_id', 'profile_type', 'avg_transaction_amount', 'daily_transaction_rate', 'home_city', 'home_lat', 'home_lon', 'registered_device', 'amount_vs_user_avg', 'amount_deviation', 'amount_ratio', 'is_new_device', 'distance_from_home_km', 'transactions_last_10m', 'transactions_last_1h', 'transactions_last_24h', 'time_since_previous_transaction', 'recipient_transaction_count', 'is_new_recipient', 'device_transaction_count', 'location_transaction_count', 'is_new_location', 'distance_from_previous_location_km', 'travel_speed_kmph', 'impossible_travel_flag', 'amount_vs_user_std', 'extreme_amount_flag', 'historical_avg_amount', '

In [63]:
print(df.head())

  transaction_id           timestamp sender_id   receiver_id   amount  \
0      TX0000106 2026-01-03 13:26:04   U100000  R_FRAUD_1058  2582.92   
1      TX0000129 2026-01-05 01:51:05   U100000        R88146  1327.52   
2      TX0000118 2026-01-14 06:21:25   U100000        R90967  1007.41   
3      TX0000045 2026-01-15 14:05:07   U100000        R25509   663.43   
4      TX0000130 2026-01-16 07:27:50   U100000        R54357   800.96   

  merchant_category transaction_type       city  latitude  longitude  \
0         Education              P2M  Hyderabad  17.39472   78.48280   
1            Travel              P2M  Hyderabad  17.44129   78.51521   
2              Food              P2M  Hyderabad  17.38443   78.45662   
3              Food              P2M  Hyderabad  17.43150   78.46320   
4         Utilities              P2M  Hyderabad  17.35805   78.50146   

  device_id      upi_channel  fraud_label     fraud_scenario        date  \
0   D100000               QR            1  repeated_

In [64]:
df.to_csv(r"D:\Machine Learning\upi_fraud_detection\data\processed\feature_engineered_transactions.csv", index=False)

print("Feature engineered dataset saved successfully.")

Feature engineered dataset saved successfully.
